# Experiment 2 - Translate-Train (English -> Bengali)

Fine-tune on English posts translated to **Bengali** and evaluate on native Bengali.

| Condition | Backbone | Train | Validation | Test |
|---|---|---|---|---|
| **2** translate-train | `FacebookAI/xlm-roberta-base` | en-train -> bn | en-val -> bn | bn-test (native) |

The split seed is 42, matching experiments 1 and 4. Three training seeds (42, 43, 44) measure training variability on those fixed splits.

Models are pushed to a **private Hugging Face Hub repo** after every run.

In [ ]:
# Runtime check: this experiment needs a GPU.
import torch

print("torch         :", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. In Colab select Runtime > Change runtime type > GPU.")
print("device        :", torch.cuda.get_device_name(0))

In [ ]:
# Install pinned deps. torch is deliberately NOT installed so Colab's
# CUDA build stays intact.
!pip install -q "transformers==5.18.0" accelerate openpyxl pandas scikit-learn

In [ ]:
# Get the project. Idempotent: safe to re-run after a runtime restart.
# The datasets are tracked in the repo, so the clone brings code + data.
import os, subprocess, pathlib

if not pathlib.Path("NLP").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/iammatthi/NLP.git"], check=True)
os.chdir("NLP")
print("working dir:", os.getcwd())

for rel in (
    "data/raw/depression-severity-english.csv",
    "data/raw/depression-severity-bengali.xlsx",
    "data/translated/en_to_bn.csv",
):
    data_file = pathlib.Path(rel)
    assert data_file.exists(), f"{data_file} missing - the clone did not bring the data"
    print("dataset OK:", data_file, f"({data_file.stat().st_size:,} bytes)")

In [ ]:
# Persist trained checkpoints to a private Hugging Face Hub repo.
# Set HF_TOKEN in Colab Secrets (key icon, left sidebar) - it is read
# from there and never printed. If no secret is set, you get a login prompt.
import os

HF_REPO = "simonfrnd/nlp_experiment2"   # experiment 2 writes to this repo
print("HF_REPO =", HF_REPO)

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    print("HF_TOKEN: read from Colab Secrets")
except Exception:
    print("HF_TOKEN: no Colab Secret found, falling back to a login prompt")

from huggingface_hub import login, whoami
if token:
    login(token=token)
else:
    from huggingface_hub import notebook_login
    notebook_login()

print("authenticated as:", whoami()["name"])
print("keep the repo private: the source datasets state no licence")

In [ ]:
# CPU-only build check: verifies XLM-R loads and produces (n, 4) logits.
!python scripts/smoke_test_model.py --models FacebookAI/xlm-roberta-base

In [ ]:
# Stratified 80/10/10 splits, fixed seed. Regenerated every session; the seed
# makes them byte-reproducible so runs stay comparable.
# Both languages are needed: en for training/validation, bn for test evaluation.
!python scripts/prepare_splits.py --lang bn --seed 42
!python scripts/prepare_translated_splits.py --translations data/translated/en_to_bn.csv --seed 42

## Train

Class-weighted cross-entropy is on by default (inverse-frequency from the translated
English train split) to counteract the label prior without resampling. Checkpoints are
selected on translated English validation macro-F1; the best one is written to `<output-dir>/final/`
and pushed to the Hub immediately.

Each run takes ~15-20 min on a T4.

In [ ]:
%%bash
set -e
# Condition 2 - XLM-R on translated English
python train.py --model xlmr --lang en --splits-dir data/splits/en_to_bn --output-dir outputs/2_xlmr_seed42 --seed 42 --epochs 10 --patience 4 --warmup-steps 100 --train-batch-size 16 --learning-rate 2e-5 --bf16 && python scripts/push_checkpoint.py --repo-id simonfrnd/nlp_experiment2 --run-dir outputs/2_xlmr_seed42
python train.py --model xlmr --lang en --splits-dir data/splits/en_to_bn --output-dir outputs/2_xlmr_seed43 --seed 43 --epochs 10 --patience 4 --warmup-steps 100 --train-batch-size 16 --learning-rate 2e-5 --bf16 && python scripts/push_checkpoint.py --repo-id simonfrnd/nlp_experiment2 --run-dir outputs/2_xlmr_seed43
python train.py --model xlmr --lang en --splits-dir data/splits/en_to_bn --output-dir outputs/2_xlmr_seed44 --seed 44 --epochs 10 --patience 4 --warmup-steps 100 --train-batch-size 16 --learning-rate 2e-5 --bf16 && python scripts/push_checkpoint.py --repo-id simonfrnd/nlp_experiment2 --run-dir outputs/2_xlmr_seed44

## Evaluate on the held-out test split

In [ ]:
%%bash
set -e
python evaluate.py --model-dir outputs/2_xlmr_seed42/final --lang bn --split test --seed 42
python evaluate.py --model-dir outputs/2_xlmr_seed43/final --lang bn --split test --seed 42
python evaluate.py --model-dir outputs/2_xlmr_seed44/final --lang bn --split test --seed 42

# metrics are tiny - push them across too
python scripts/push_checkpoint.py --repo-id simonfrnd/nlp_experiment2 --run-dir outputs/2_xlmr_seed42 --metrics-only
python scripts/push_checkpoint.py --repo-id simonfrnd/nlp_experiment2 --run-dir outputs/2_xlmr_seed43 --metrics-only
python scripts/push_checkpoint.py --repo-id simonfrnd/nlp_experiment2 --run-dir outputs/2_xlmr_seed44 --metrics-only

## Summarise across seeds

In [ ]:
import glob, json, statistics as st

def mean_std(vals):
    m = st.mean(vals)
    s = st.stdev(vals) if len(vals) > 1 else 0.0
    return f"{m:.4f} +/- {s:.4f}"

runs = [json.load(open(p)) for p in sorted(glob.glob("outputs/2_xlmr_seed*/final/eval_test.json"))]
assert runs, "no results - run the training and evaluation cells first"
print(f"2_xlmr  n={len(runs)} (translate-train)")
for k in ("macro_f1", "qwk", "mae", "accuracy"):
    print(f"  {k:18s} {mean_std([r[k] for r in runs])}")
for k in ("upgrade_rate", "downgrade_rate", "error_rate", "mean_signed_error"):
    print(f"  {k:18s} {mean_std([r['direction'][k] for r in runs])}")

print("\nper-class F1 (mean across seeds):")
for cls in ("minimal", "mild", "moderate", "severe"):
    print(f"  {cls:9s} {mean_std([r[cls] for r in runs])}")

In [ ]:
# Export metrics to the local machine. Checkpoints are large and live on the Hub, this zip is just the small numbers.
import glob, zipfile

paths = sorted(
    glob.glob("outputs/2_xlmr_seed*/final/eval_test.json")
    + glob.glob("outputs/2_xlmr_seed*/train_metrics.json")
)
assert paths, "no results found - run the training and evaluation cells first"

with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p)

print(f"zipped {len(paths)} files:")
for p in paths:
    print(" ", p)

from google.colab import files
files.download("results.zip")
print("downloaded results.zip")

In [ ]:
# What ended up on the HF. Expect 3 run directories (~3.3 GB total).
# Only the best checkpoint per run is kept; intermediate checkpoint-* folders are deliberately not uploaded.
from huggingface_hub import HfApi
api = HfApi()
info = api.repo_info(HF_REPO, files_metadata=True)
print(f"repo: {info.id}  (private={info.private})")
total = 0
for s in sorted(info.siblings, key=lambda s: s.rfilename):
    size = getattr(s, "size", 0) or 0
    total += size
    print(f"  {size/1e6:8.1f} MB  {s.rfilename}")
print(f"  total {total/1e6:.1f} MB")